#Data Ingestion (JDBC)

###User Data

In [0]:
from pyspark.sql.types import StructType, StructField, MapType, StringType
from pyspark.sql.functions import explode

catlog = "jrvs_databricks_fundamentals"
folder = f"{CATALOG}.bronze" 

URL = ("jdbc:sqlserver://jarvis-test.database.windows.net:1433;"
       "database=tictest;encrypt=true;trustServerCertificate=false;"
       "hostNameInCertificate=*.database.windows.net;loginTimeout=30;")

def read_jdbc(table):
    return (spark.read.format("jdbc")
        .option("url", URL)
        .option("dbtable", table)
        .option("user", "jarvis-admin@jarvis-test")
        .option("password", "Wilson@99") #note: bad practice but dont hardcode 
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load())

In [ ]:
read_jdbc("dbo.users_data").write.mode("overwrite").saveAsTable(f"{folder}.users_data")
read_jdbc("dbo.transactions_data").write.mode("overwrite").saveAsTable(f"{folder}.transactions_data")
read_jdbc("dbo.cards_data").write.mode("overwrite").saveAsTable(f"{folder}.cards_data")

#Data Ingestion (Azure Storage)

In [ ]:
from pyspark.sql.functions import from_json, explode, col
from pyspark.sql.types import MapType, StringType, StructType, StructField

# source JSON files sit in ADLS read them through the governed Volume path
volume = "/Volumes/jrvs_databricks_fundamentals/bronze/volume"  
bronze = "jrvs_databricks_fundamentals.bronze"                  

mcc_schema = MapType(StringType(), StringType())

mcc = (
    spark.read.text(f"{volume}/mcc_codes.json", wholetext=True) # 1 row, value = whole file as a string
    .select(from_json(col("value"), mcc_schema).alias("m")) # parse string -> map column "m"
    .select(explode("m").alias("mcc", "mcc_description")) # explode map -> one row per pair
)
mcc.write.mode("overwrite").saveAsTable(f"{bronze}.mcc_codes")

fraud_schema = StructType([StructField("target", MapType(StringType(), StringType()), True)])

fraud_raw = (spark.read.option("multiLine", "true").schema(fraud_schema)
             .json(f"{volume}/train_fraud_labels.json")
             )
fraud_df  = fraud_raw.select(explode("target").alias("transaction_id", "label"))
fraud_df.write.mode("overwrite").saveAsTable(f"{bronze}.fraud_labels")